# 01 — PDF collection inventory and data understanding

Run this notebook from the `notebooks/` folder. It scans every PDF under `../pdfs` (including subfolders) and saves CSVs under `../output/inventory`. It does **not** fetch Publinova records, clean text, or generate keywords. The PDF stays local.

The scan produces candidates and quality measurements. A DOI or date candidate is not a verified publication fact. Put reviewed decisions in `documents.csv`; add manually checked Publinova/PDF facts to the observation and term CSVs using `source_type=manual_review` or `publinova_record`. Reruns retain those entries.

**Before rerunning:** close CSVs in Excel, which may lock them on Windows. Keep a backup before making substantial manual edits. Do not edit `document_id`, `file_sha256`, or rows labelled `pdf_embedded` / `pdf_text`, because those rows are rebuilt.


## 1. Dependencies and folders

Install once with `%pip install pymupdf pandas` if either import is missing. All paths are resolved relative to this notebook's working directory; check the displayed paths before scanning.


In [3]:
from pathlib import Path
import hashlib
import re
import pymupdf
import pandas as pd
from IPython.display import display

#change the Path() to ../pdfs if you get an error
PDF_DIR = Path('../pdfs').resolve()
OUTPUT_DIR = Path('../output/inventory').resolve()
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
if not PDF_DIR.is_dir():
    raise FileNotFoundError(f'Create the PDF folder first: {PDF_DIR}')
pdf_paths = sorted(PDF_DIR.rglob('*.pdf'), key=lambda p: str(p).lower())
print(f'PDF folder: {PDF_DIR}\nCSV folder: {OUTPUT_DIR}\nPDFs found: {len(pdf_paths)}')


PDF folder: /Users/georgyfurs/Documents/ICT_Year4/CPS/DDD/pdfs
CSV folder: /Users/georgyfurs/Documents/ICT_Year4/CPS/DDD/output/inventory
PDFs found: 6


## 2. Schema

- **`documents.csv`**: one row per unique PDF *content*. `document_id` is based on SHA-256, so an unchanged file keeps the same ID if moved. Duplicate copies are listed separately in the final duplicate summary. `pdf_path` is relative to `pdfs/`.
- **`metadata_observations.csv`**: one candidate value per row, with its source and location. Automatic candidates are rebuilt; manually added `manual_review` and `publinova_record` rows survive.
- **`document_terms.csv`**: one supplied keyword, theme or author per row. This notebook reads only embedded PDF keywords automatically. Add author names after checking them; a contact person or a Publinova “People” listing is not necessarily an author list.

Leave unknown fields empty. `publication_year` is a four-digit year; `publication_date` and `online_date` are ISO `YYYY-MM-DD` only when a full date has been checked. Do not substitute the PDF creation date for publication date. `text_status` means `extractable` (text on all pages), `partial` (some pages empty), `no_text` (no extracted text), or `error`. An empty page might be a cover or scan, so it is not automatically marked “scanned”.


In [5]:
DOCUMENT_COLUMNS = [
    'document_id', 'file_sha256', 'file_name', 'pdf_path', 'file_size_bytes',
    'page_count', 'extracted_char_count', 'pages_without_text', 'text_status',
    'scan_error', 'publinova_url', 'title', 'doi', 'publication_year',
    'publication_date', 'online_date', 'document_type', 'language',
    'metadata_review_status', 'review_notes'
]
REVIEW_COLUMNS = [
    'publinova_url', 'title', 'doi', 'publication_year', 'publication_date',
    'online_date', 'document_type', 'language', 'metadata_review_status',
    'review_notes'
]
OBS_COLUMNS = [
    'document_id', 'field_name', 'observed_value', 'source_type',
    'source_location', 'extraction_method', 'review_status'
]
TERM_COLUMNS = [
    'document_id', 'term_type', 'term_value', 'source_type',
    'position', 'review_status'
]

def load_existing(name, columns):
    path = OUTPUT_DIR / name
    if not path.exists():
        return pd.DataFrame(columns=columns)
    df = pd.read_csv(path, dtype=str, keep_default_na=False, encoding='utf-8-sig')
    missing = set(columns) - set(df.columns)
    if missing:
        raise ValueError(f'{path} is missing columns: {sorted(missing)}')
    return df[columns]

old_documents = load_existing('documents.csv', DOCUMENT_COLUMNS)
old_observations = load_existing('metadata_observations.csv', OBS_COLUMNS)
old_terms = load_existing('document_terms.csv', TERM_COLUMNS)
print('Existing reviewed rows will be retained where their PDF still exists.')


Existing reviewed rows will be retained where their PDF still exists.


## 3. Scan PDFs and collect source-specific candidates

The regexes deliberately gather only simple DOI and “Available online” candidates from the opening pages. They are not general-purpose author or publication-date extraction. `pdf_embedded` values can also be incorrect.


In [7]:
def sha256_file(path):
    digest = hashlib.sha256()
    with path.open('rb') as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()

def add_observation(rows, document_id, field, value, source, location, method):
    if value is not None and str(value).strip():
        rows.append({
            'document_id': document_id, 'field_name': field,
            'observed_value': str(value).strip(), 'source_type': source,
            'source_location': location, 'extraction_method': method,
            'review_status': 'unreviewed'
        })

documents, observations, terms, duplicate_paths = [], [], [], []
seen_hashes = set()
for path in pdf_paths:
    digest = sha256_file(path)
    if digest in seen_hashes:
        duplicate_paths.append({'file_name': path.name, 'file_sha256': digest})
        continue
    seen_hashes.add(digest)
    document_id = 'doc_' + digest[:16]
    row = dict.fromkeys(DOCUMENT_COLUMNS, '')
    row.update(document_id=document_id, file_sha256=digest,
               file_name=path.name, pdf_path=path.relative_to(PDF_DIR).as_posix(),
               file_size_bytes=path.stat().st_size,
               metadata_review_status='unreviewed')
    try:
        with pymupdf.open(path) as pdf:
            info = pdf.metadata or {}
            counts = [len(page.get_text('text').strip()) for page in pdf]
            opening = '\n'.join(pdf[i].get_text('text') for i in range(min(3, len(pdf))))
            row['page_count'] = len(pdf)
            row['extracted_char_count'] = sum(counts)
            row['pages_without_text'] = sum(n == 0 for n in counts)
            row['text_status'] = ('no_text' if not any(counts) else
                                  'partial' if any(n == 0 for n in counts) else 'extractable')
            for field, key in [('title', 'title'), ('author', 'author'),
                               ('subject', 'subject'), ('pdf_keywords', 'keywords'),
                               ('pdf_creation_date', 'creationDate')]:
                add_observation(observations, document_id, field, info.get(key),
                                'pdf_embedded', f'PDF metadata/{key}', 'embedded')
            doi = re.search(r'(?i)\b10\.\d{4,9}/[-._;()/:A-Z0-9]+', opening)
            if doi:
                add_observation(observations, document_id, 'doi',
                                doi.group(0).rstrip('.,;'), 'pdf_text',
                                'PDF pages 1–3', 'regex')
            online = re.search(r'(?i)Available online\s+(\d{1,2}\s+[A-Za-z]+\s+\d{4})', opening)
            if online:
                add_observation(observations, document_id, 'online_date',
                                online.group(1), 'pdf_text', 'PDF pages 1–3', 'regex')
            keywords = info.get('keywords') or ''
            for pos, keyword in enumerate(re.split(r'[;,]', keywords), start=1):
                if keyword.strip():
                    terms.append({'document_id': document_id, 'term_type': 'keyword',
                                  'term_value': keyword.strip(), 'source_type': 'pdf_embedded',
                                  'position': pos, 'review_status': 'unreviewed'})
    except Exception as exc:
        row['text_status'] = 'error'
        row['scan_error'] = f'{type(exc).__name__}: {exc}'[:300]
    documents.append(row)

new_documents = pd.DataFrame(documents, columns=DOCUMENT_COLUMNS)
new_observations = pd.DataFrame(observations, columns=OBS_COLUMNS)
new_terms = pd.DataFrame(terms, columns=TERM_COLUMNS)
print(f'Unique PDFs: {len(new_documents)}; duplicate copies: {len(duplicate_paths)}; extraction errors: {(new_documents.text_status == "error").sum()}')


Unique PDFs: 3; duplicate copies: 3; extraction errors: 0


## 4. Preserve manual review and save

Only columns intended for human decisions are carried forward in `documents.csv`. Only observation/term rows with `source_type=manual_review` or `publinova_record` are carried forward. Removed PDFs disappear from the current inventory; keep backups if you want a historical audit trail.


In [9]:
if old_documents.duplicated('document_id').any():
    raise ValueError('Existing documents.csv has duplicate document_id values. Resolve these before rerunning.')
if not old_documents.empty:
    review_by_id = old_documents.set_index('document_id')[REVIEW_COLUMNS]
    for idx, document_id in new_documents.document_id.items():
        if document_id in review_by_id.index:
            for field in REVIEW_COLUMNS:
                value = review_by_id.at[document_id, field]
                if value:
                    new_documents.at[idx, field] = value

active_ids = set(new_documents.document_id)
manual_sources = {'manual_review', 'publinova_record'}
kept_obs = old_observations[
    old_observations.source_type.isin(manual_sources) &
    old_observations.document_id.isin(active_ids)
]
kept_terms = old_terms[
    old_terms.source_type.isin(manual_sources) &
    old_terms.document_id.isin(active_ids)
]
all_observations = pd.concat([new_observations, kept_obs], ignore_index=True)
all_terms = pd.concat([new_terms, kept_terms], ignore_index=True)
for name, frame in [
    ('documents.csv', new_documents),
    ('metadata_observations.csv', all_observations),
    ('document_terms.csv', all_terms),
]:
    target = OUTPUT_DIR / name
    frame.to_csv(target, index=False, encoding='utf-8-sig')
    print(f'Saved {target.name}: {len(frame)} rows')


Saved documents.csv: 3 rows
Saved metadata_observations.csv: 12 rows
Saved document_terms.csv: 6 rows


## 5. Data-understanding overview

These summaries help choose which documents need manual review. Empty selected fields mean *unreviewed or unknown*, not necessarily missing from the PDF. Check candidate coverage in the observations table.


In [11]:
display(new_documents[['file_name', 'page_count', 'extracted_char_count',
                       'pages_without_text', 'text_status', 'metadata_review_status']])
print('Text status:')
display(new_documents.text_status.value_counts(dropna=False).rename_axis('status').to_frame('PDFs'))
print('Candidate coverage by field (unique PDFs):')
if all_observations.empty:
    print('No metadata candidates found yet.')
else:
    display(all_observations.groupby('field_name').document_id.nunique()
            .sort_values(ascending=False).rename('PDFs').to_frame())
print('Review queue:')
display(new_documents.loc[(new_documents.text_status != 'extractable') |
                          (new_documents.metadata_review_status != 'verified'),
                          ['document_id', 'file_name', 'text_status', 'scan_error',
                           'metadata_review_status', 'review_notes']])
if duplicate_paths:
    print('Identical duplicate files skipped:')
    display(pd.DataFrame(duplicate_paths))


,file_name,page_count,extracted_char_count,pages_without_text,text_status,metadata_review_status
0,Animals on the leeve-checkpoint.pdf,22,5084,0,extractable,unreviewed
1,Business Case Vlissingen-checkpoint.pdf,37,89974,0,extractable,unreviewed
2,Nannochloropsis-checkpoint.pdf,15,90983,0,extractable,unreviewed


Text status:


,PDFs
status,
extractable,3


Candidate coverage by field (unique PDFs):


,PDFs
field_name,
author,3
pdf_creation_date,3
title,2
doi,1
online_date,1
pdf_keywords,1
subject,1


Review queue:


,document_id,file_name,text_status,scan_error,metadata_review_status,review_notes
0,doc_8b361ecbae1548b4,Animals on the leeve-checkpoint.pdf,extractable,,unreviewed,
1,doc_1f92b33dfeccc1c9,Business Case Vlissingen-checkpoint.pdf,extractable,,unreviewed,
2,doc_4d0489e9657d8e45,Nannochloropsis-checkpoint.pdf,extractable,,unreviewed,


Identical duplicate files skipped:


,file_name,file_sha256
0,Animals on the leeve.pdf,8b361ecbae1548b4f3e789b1a8ab06dd7f6688f2369ea1...
1,Business Case Vlissingen.pdf,1f92b33dfeccc1c9975f1055cdc259fde4fba70efd0599...
2,Nannochloropsis.pdf,4d0489e9657d8e45b31a0da24917ad8fcf018b843c4fc2...
